# 🏦 Microsoft Fabric — Demo End-to-End : Retail Banking × Assurance (Pacifica)

> **Architecture multi-domaines avec OneLake Shortcuts, RLS dynamique, Direct Lake**  
> **Tenant demo** : `MngEnvMCAP578215.onmicrosoft.com`  
> **Date** : Juin 2026  

---

## 📋 Plan de la démonstration

| Étape | Contenu |
|---|---|
| **0** | Architecture & diagramme textuel |
| **1** | Génération des données sample (Python) |
| **2** | Simulation du chargement dans les Lakehouses |
| **3** | Shortcuts OneLake — configuration & permissions |
| **4** | Row-Level Security (RLS) — règles DAX |
| **5** | Object-Level Security (OLS) / Column-Level Security |
| **6** | Semantic Model — schéma en étoile |
| **7** | Simulation de requêtes par utilisateur |
| **8** | Tests de sécurité — validation de l'isolation |
| **9** | Extension — Assurance Vie / Gestion de Patrimoine |

---

## ⚠️ Points de sécurité critiques

> **OneLake RLS** est la couche de contrôle primaire car elle s'applique à **tous les moteurs**  
> (Spark, SQL Analytics Endpoint, Power BI Direct Lake).  
> Le RLS du Semantic Model Power BI est une couche **complémentaire** pour le filtrage métier dynamique.  
> **Ne jamais compter uniquement sur le RLS Semantic Model** — il ne protège pas les accès SQL directs.

## 🗺️ Étape 0 — Architecture Fabric : Diagramme textuel

```
╔══════════════════════════════════════════════════════════════════════════════════╗
║  MICROSOFT FABRIC TENANT — MngEnvMCAP578215.onmicrosoft.com                     ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                  ║
║  ┌──────────────────────────────┐    OneLake    ┌────────────────────────────┐  ║
║  │  WS-Banking (LCL / CADIF)    │   Shortcuts   │  WS-Insurance (Pacifica)   │  ║
║  │  Capacité F64                │──────────────►│  Capacité F64              │  ║
║  │                              │               │                            │  ║
║  │  📦 Lakehouse_Banking        │               │  📦 Lakehouse_Insurance    │  ║
║  │  ┌──────────────────────┐   │               │  ┌──────────────────────┐  │  ║
║  │  │ Δ dim_customers       │──────────────────►│  │ ⤷ sc_customers       │  │  ║
║  │  │ Δ fact_bank_accounts  │──────────────────►│  │ ⤷ sc_bank_accounts   │  │  ║
║  │  │ Δ bridge_ins_cust     │──────────────────►│  │ ⤷ sc_bridge_ins      │  │  ║
║  │  │ Δ dim_advisors        │   │               │  │                      │  │  ║
║  │  └──────────────────────┘   │               │  │ Δ insurance_contracts│  │  ║
║  │                              │               │  │ Δ insurance_claims   │  │  ║
║  │  🔒 SQL Endpoint             │               │  │ Δ security_table     │  │  ║
║  │  (Banking users only)        │               │  └──────────────────────┘  │  ║
║  └──────────────────────────────┘               │                            │  ║
║                                                  │  🧠 Semantic Model         │  ║
║                                                  │  (Direct Lake)             │  ║
║                                                  │  RLS: USERPRINCIPALNAME()  │  ║
║                                                  │  OLS: PII columns hidden   │  ║
║                                                  └────────────────────────────┘  ║
║                                                                                  ║
║  ┌─────────────────────────────────────────────────────────────────────────┐    ║
║  │  Entra ID Groups & Users                                                 │    ║
║  │  GRP-Banking-Analysts → WS-Banking (Contributor)                        │    ║
║  │  GRP-Insurance-Analysts → WS-Insurance (Contributor)                    │    ║
║  │  advisor1@bank.com   → Security_Table[customers: A, B]                  │    ║
║  │  advisor2@bank.com   → Security_Table[customers: C]                     │    ║
║  │  insurance@pacifica.com → Security_Table[insurance customers only]      │    ║
║  └─────────────────────────────────────────────────────────────────────────┘    ║
╚══════════════════════════════════════════════════════════════════════════════════╝
```

### 🔑 Flux de sécurité par couche

| Couche | Mécanisme | Portée | Moteurs protégés |
|---|---|---|---|
| **Workspace** | Rôles Fabric | Qui peut voir le workspace | Tous |
| **Lakehouse item** | Item permissions | Qui peut lire le Lakehouse | Tous |
| **SQL Endpoint** | GRANT/DENY T-SQL | Accès SQL direct | SQL uniquement |
| **OneLake RLS** | Row filters sur Delta | Filtrage au niveau fichier | Spark + SQL + PBI ✅ |
| **Semantic Model RLS** | Règles DAX | Filtrage rapport Power BI | Power BI uniquement |
| **OLS / CLS** | Semantic Model | Tables/colonnes masquées | Power BI uniquement |

## 📦 Étape 1 — Génération des données sample

Nous allons générer un dataset réaliste avec :
- **10 clients** (dont 5 avec contrat assurance)
- **3 utilisateurs** avec des périmètres différents
- **Une Security_Table** pour le RLS dynamique

Exécutez la cellule suivante pour générer et afficher toutes les tables.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# ─────────────────────────────────────────────
# TABLE 1 : dim_customers  (Banking Lakehouse)
# ─────────────────────────────────────────────
dim_customers = pd.DataFrame([
    {"customer_id": "CUS-001", "name": "Marie Dupont",    "region": "Île-de-France", "segment": "PREMIUM",  "advisor_email": "advisor1@bank.com"},
    {"customer_id": "CUS-002", "name": "Jean Martin",     "region": "Occitanie",     "segment": "STANDARD", "advisor_email": "advisor1@bank.com"},
    {"customer_id": "CUS-003", "name": "Sophie Leroy",    "region": "Auvergne-RA",   "segment": "YOUNG",    "advisor_email": "advisor2@bank.com"},
    {"customer_id": "CUS-004", "name": "Pierre Bernard",  "region": "PACA",          "segment": "PREMIUM",  "advisor_email": "advisor2@bank.com"},
    {"customer_id": "CUS-005", "name": "Isabelle Moreau", "region": "Bretagne",      "segment": "STANDARD", "advisor_email": "advisor1@bank.com"},
    {"customer_id": "CUS-006", "name": "Thomas Simon",    "region": "Normandie",     "segment": "YOUNG",    "advisor_email": "advisor2@bank.com"},
    {"customer_id": "CUS-007", "name": "Claire Laurent",  "region": "Grand Est",     "segment": "PREMIUM",  "advisor_email": "advisor1@bank.com"},
    {"customer_id": "CUS-008", "name": "François Petit",  "region": "Hauts-de-Fr",   "segment": "STANDARD", "advisor_email": "advisor2@bank.com"},
    {"customer_id": "CUS-009", "name": "Nathalie Garcia", "region": "Nouvelle-Aq",   "segment": "STANDARD", "advisor_email": "advisor1@bank.com"},
    {"customer_id": "CUS-010", "name": "Luc Roux",        "region": "Pays de Loire", "segment": "PREMIUM",  "advisor_email": "advisor2@bank.com"},
])

# ─────────────────────────────────────────────
# TABLE 2 : fact_bank_accounts  (Banking Lakehouse)
# ─────────────────────────────────────────────
fact_bank_accounts = pd.DataFrame([
    {"account_id": "ACC-001", "customer_id": "CUS-001", "product_type": "CHECKING", "balance": 12450.00},
    {"account_id": "ACC-002", "customer_id": "CUS-001", "product_type": "SAVINGS",  "balance": 45000.00},
    {"account_id": "ACC-003", "customer_id": "CUS-002", "product_type": "CHECKING", "balance": 3200.00},
    {"account_id": "ACC-004", "customer_id": "CUS-003", "product_type": "CHECKING", "balance": 1800.00},
    {"account_id": "ACC-005", "customer_id": "CUS-004", "product_type": "LOAN",     "balance": -25000.00},
    {"account_id": "ACC-006", "customer_id": "CUS-004", "product_type": "SAVINGS",  "balance": 78000.00},
    {"account_id": "ACC-007", "customer_id": "CUS-005", "product_type": "CHECKING", "balance": 5600.00},
    {"account_id": "ACC-008", "customer_id": "CUS-006", "product_type": "CHECKING", "balance": 920.00},
    {"account_id": "ACC-009", "customer_id": "CUS-007", "product_type": "SAVINGS",  "balance": 120000.00},
    {"account_id": "ACC-010", "customer_id": "CUS-008", "product_type": "CHECKING", "balance": 4100.00},
    {"account_id": "ACC-011", "customer_id": "CUS-009", "product_type": "CHECKING", "balance": 7800.00},
    {"account_id": "ACC-012", "customer_id": "CUS-010", "product_type": "SAVINGS",  "balance": 95000.00},
])

# ─────────────────────────────────────────────
# TABLE 3 : bridge_ins_customers  (Banking — liste blanche partagée avec Insurance)
# ─────────────────────────────────────────────
bridge_ins_customers = pd.DataFrame([
    {"bridge_id": "BRG-001", "customer_id": "CUS-001", "insurance_consent": True,  "sharing_scope": "FULL"},
    {"bridge_id": "BRG-002", "customer_id": "CUS-002", "insurance_consent": True,  "sharing_scope": "BASIC"},
    {"bridge_id": "BRG-003", "customer_id": "CUS-004", "insurance_consent": True,  "sharing_scope": "FULL"},
    {"bridge_id": "BRG-004", "customer_id": "CUS-007", "insurance_consent": True,  "sharing_scope": "FULL"},
    {"bridge_id": "BRG-005", "customer_id": "CUS-009", "insurance_consent": True,  "sharing_scope": "BASIC"},
])

# ─────────────────────────────────────────────
# TABLE 4 : insurance_contracts  (Insurance Lakehouse — native)
# ─────────────────────────────────────────────
insurance_contracts = pd.DataFrame([
    {"contract_id": "CTR-001", "customer_id": "CUS-001", "contract_type": "MRH",  "product_label": "Multirisque Habitation", "premium": 380.00,  "status": "ACTIVE"},
    {"contract_id": "CTR-002", "customer_id": "CUS-001", "contract_type": "AUTO", "product_label": "Assurance Auto",         "premium": 820.00,  "status": "ACTIVE"},
    {"contract_id": "CTR-003", "customer_id": "CUS-002", "contract_type": "AUTO", "product_label": "Assurance Auto",         "premium": 650.00,  "status": "ACTIVE"},
    {"contract_id": "CTR-004", "customer_id": "CUS-004", "contract_type": "VIE",  "product_label": "Assurance Vie",          "premium": 1200.00, "status": "ACTIVE"},
    {"contract_id": "CTR-005", "customer_id": "CUS-007", "contract_type": "PREV", "product_label": "Prévoyance",             "premium": 540.00,  "status": "ACTIVE"},
    {"contract_id": "CTR-006", "customer_id": "CUS-009", "contract_type": "MRH",  "product_label": "Multirisque Habitation", "premium": 290.00,  "status": "ACTIVE"},
])

# ─────────────────────────────────────────────
# TABLE 5 : security_table  (Insurance Semantic Model — RLS mapping)
# ─────────────────────────────────────────────
# Maps each user to the customers they are allowed to see
security_table = pd.DataFrame([
    # advisor1@bank.com → clients CUS-001, CUS-002, CUS-005, CUS-007, CUS-009
    {"user_email": "advisor1@bank.com",           "customer_id": "CUS-001"},
    {"user_email": "advisor1@bank.com",           "customer_id": "CUS-002"},
    {"user_email": "advisor1@bank.com",           "customer_id": "CUS-005"},
    {"user_email": "advisor1@bank.com",           "customer_id": "CUS-007"},
    {"user_email": "advisor1@bank.com",           "customer_id": "CUS-009"},
    # advisor2@bank.com → clients CUS-003, CUS-004, CUS-006, CUS-008, CUS-010
    {"user_email": "advisor2@bank.com",           "customer_id": "CUS-003"},
    {"user_email": "advisor2@bank.com",           "customer_id": "CUS-004"},
    {"user_email": "advisor2@bank.com",           "customer_id": "CUS-006"},
    {"user_email": "advisor2@bank.com",           "customer_id": "CUS-008"},
    {"user_email": "advisor2@bank.com",           "customer_id": "CUS-010"},
    # insurance_user@pacifica.com → uniquement les clients avec contrat assurance actif
    {"user_email": "insurance_user@pacifica.com", "customer_id": "CUS-001"},
    {"user_email": "insurance_user@pacifica.com", "customer_id": "CUS-002"},
    {"user_email": "insurance_user@pacifica.com", "customer_id": "CUS-004"},
    {"user_email": "insurance_user@pacifica.com", "customer_id": "CUS-007"},
    {"user_email": "insurance_user@pacifica.com", "customer_id": "CUS-009"},
])

# ─────────────────────────────────────────────
# TABLE 6 : insurance_claims  (Insurance Lakehouse — native)
# ─────────────────────────────────────────────
insurance_claims = pd.DataFrame([
    {"claim_id": "CLM-001", "contract_id": "CTR-001", "claim_date": "2024-03-15", "claim_type": "Dégât des eaux",    "amount": 2400.00, "status": "CLOSED"},
    {"claim_id": "CLM-002", "contract_id": "CTR-002", "claim_date": "2024-07-22", "claim_type": "Accident auto",    "amount": 8700.00, "status": "OPEN"},
    {"claim_id": "CLM-003", "contract_id": "CTR-003", "claim_date": "2025-01-10", "claim_type": "Bris de glace",    "amount": 450.00,  "status": "CLOSED"},
    {"claim_id": "CLM-004", "contract_id": "CTR-005", "claim_date": "2025-05-30", "claim_type": "Arrêt de travail", "amount": 3200.00, "status": "OPEN"},
])

print("✅ Toutes les tables ont été générées avec succès.\n")
print(f"  dim_customers        : {len(dim_customers)} lignes")
print(f"  fact_bank_accounts   : {len(fact_bank_accounts)} lignes")
print(f"  bridge_ins_customers : {len(bridge_ins_customers)} lignes")
print(f"  insurance_contracts  : {len(insurance_contracts)} lignes")
print(f"  insurance_claims     : {len(insurance_claims)} lignes")
print(f"  security_table       : {len(security_table)} lignes")

In [ ]:
# ─── Affichage des tables avec mise en forme HTML ───────────────────────────

def show_table(df, title, color="#1a73e8"):
    styled = df.style.set_table_styles([
        {"selector": "thead th", "props": f"background-color:{color};color:white;font-weight:bold;padding:6px 10px;"},
        {"selector": "tbody td", "props": "padding:5px 10px;border-bottom:1px solid #eee;"},
        {"selector": "table",    "props": "border-collapse:collapse;font-size:13px;"},
    ]).set_caption(title)
    display(styled)

print("=" * 70)
print("  BANKING WORKSPACE — Lakehouse_Banking")
print("=" * 70)
show_table(dim_customers,      "📋 dim_customers (Banking — source de vérité client)", "#1565c0")
show_table(fact_bank_accounts, "💳 fact_bank_accounts (Banking — comptes)", "#1565c0")
show_table(bridge_ins_customers, "🔗 bridge_ins_customers (Banking — liste blanche Assurance)", "#6a1b9a")

print("=" * 70)
print("  INSURANCE WORKSPACE — Lakehouse_Insurance")
print("=" * 70)
show_table(insurance_contracts, "📄 insurance_contracts (Insurance — natif)", "#2e7d32")
show_table(insurance_claims,    "🚨 insurance_claims (Insurance — natif)", "#2e7d32")

print("=" * 70)
print("  SEMANTIC MODEL — Security_Table (RLS dynamique)")
print("=" * 70)
show_table(security_table, "🔐 security_table — Mapping utilisateur → clients autorisés", "#b71c1c")

## 💾 Étape 2 — Chargement dans les Lakehouses Fabric

### Distribution des tables par Lakehouse

| Table | Lakehouse | Type | Accès shortcut |
|---|---|---|---|
| `dim_customers` | **Lakehouse_Banking** | Delta native | ✅ Shortcutée vers Insurance |
| `fact_bank_accounts` | **Lakehouse_Banking** | Delta native | ✅ Shortcutée vers Insurance |
| `bridge_ins_customers` | **Lakehouse_Banking** | Delta native | ✅ Shortcutée vers Insurance |
| `insurance_contracts` | **Lakehouse_Insurance** | Delta native | ❌ Locale Insurance |
| `insurance_claims` | **Lakehouse_Insurance** | Delta native | ❌ Locale Insurance |
| `security_table` | **Lakehouse_Insurance** | Delta native | ❌ Locale Insurance |

> **Dans Microsoft Fabric**, le chargement se fait via un notebook Spark (PySpark).  
> La cellule ci-dessous simule le code PySpark à exécuter **dans un Fabric Notebook** attaché à chaque Lakehouse.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CODE PYSPARK À EXÉCUTER DANS UN FABRIC NOTEBOOK (Lakehouse_Banking)
#  Ce code simule le chargement des tables Banking en Delta format
# ═══════════════════════════════════════════════════════════════════════════

fabric_banking_notebook = """
# ── Fabric Notebook: Load_Banking_Tables ──────────────────────────────────
# Workspace  : WS-Banking
# Lakehouse  : Lakehouse_Banking
# ─────────────────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# ── 1. dim_customers ──────────────────────────────────────────────────────
data_customers = [
    ("CUS-001","Marie Dupont",   "Île-de-France","PREMIUM", "advisor1@bank.com"),
    ("CUS-002","Jean Martin",    "Occitanie",    "STANDARD","advisor1@bank.com"),
    ("CUS-003","Sophie Leroy",   "Auvergne-RA",  "YOUNG",   "advisor2@bank.com"),
    ("CUS-004","Pierre Bernard", "PACA",         "PREMIUM", "advisor2@bank.com"),
    ("CUS-005","Isabelle Moreau","Bretagne",     "STANDARD","advisor1@bank.com"),
    ("CUS-006","Thomas Simon",   "Normandie",    "YOUNG",   "advisor2@bank.com"),
    ("CUS-007","Claire Laurent", "Grand Est",    "PREMIUM", "advisor1@bank.com"),
    ("CUS-008","François Petit", "Hauts-de-Fr",  "STANDARD","advisor2@bank.com"),
    ("CUS-009","Nathalie Garcia","Nouvelle-Aq",  "STANDARD","advisor1@bank.com"),
    ("CUS-010","Luc Roux",       "Pays de Loire","PREMIUM", "advisor2@bank.com"),
]
schema_cust = ["customer_id","name","region","segment","advisor_email"]
df_customers = spark.createDataFrame(data_customers, schema_cust)
df_customers.write.format("delta").mode("overwrite").saveAsTable("dim_customers")
print("✅ dim_customers loaded:", df_customers.count(), "rows")

# ── 2. fact_bank_accounts ─────────────────────────────────────────────────
data_accounts = [
    ("ACC-001","CUS-001","CHECKING",12450.00),
    ("ACC-002","CUS-001","SAVINGS", 45000.00),
    ("ACC-003","CUS-002","CHECKING", 3200.00),
    ("ACC-004","CUS-003","CHECKING", 1800.00),
    ("ACC-005","CUS-004","LOAN",   -25000.00),
    ("ACC-006","CUS-004","SAVINGS", 78000.00),
    ("ACC-007","CUS-005","CHECKING", 5600.00),
    ("ACC-008","CUS-006","CHECKING",  920.00),
    ("ACC-009","CUS-007","SAVINGS",120000.00),
    ("ACC-010","CUS-008","CHECKING", 4100.00),
    ("ACC-011","CUS-009","CHECKING", 7800.00),
    ("ACC-012","CUS-010","SAVINGS", 95000.00),
]
schema_acc = ["account_id","customer_id","product_type","balance"]
df_accounts = spark.createDataFrame(data_accounts, schema_acc)
df_accounts.write.format("delta").mode("overwrite").saveAsTable("fact_bank_accounts")
print("✅ fact_bank_accounts loaded:", df_accounts.count(), "rows")

# ── 3. bridge_ins_customers ───────────────────────────────────────────────
data_bridge = [
    ("BRG-001","CUS-001",True,"FULL"),
    ("BRG-002","CUS-002",True,"BASIC"),
    ("BRG-003","CUS-004",True,"FULL"),
    ("BRG-004","CUS-007",True,"FULL"),
    ("BRG-005","CUS-009",True,"BASIC"),
]
schema_bridge = ["bridge_id","customer_id","insurance_consent","sharing_scope"]
df_bridge = spark.createDataFrame(data_bridge, schema_bridge)
df_bridge.write.format("delta").mode("overwrite").saveAsTable("bridge_ins_customers")
print("✅ bridge_ins_customers loaded:", df_bridge.count(), "rows")
"""

fabric_insurance_notebook = """
# ── Fabric Notebook: Load_Insurance_Tables ────────────────────────────────
# Workspace  : WS-Insurance
# Lakehouse  : Lakehouse_Insurance
# ─────────────────────────────────────────────────────────────────────────

# ── 4. insurance_contracts ────────────────────────────────────────────────
data_contracts = [
    ("CTR-001","CUS-001","MRH", "Multirisque Habitation", 380.00, "ACTIVE"),
    ("CTR-002","CUS-001","AUTO","Assurance Auto",          820.00, "ACTIVE"),
    ("CTR-003","CUS-002","AUTO","Assurance Auto",          650.00, "ACTIVE"),
    ("CTR-004","CUS-004","VIE", "Assurance Vie",          1200.00, "ACTIVE"),
    ("CTR-005","CUS-007","PREV","Prévoyance",              540.00, "ACTIVE"),
    ("CTR-006","CUS-009","MRH", "Multirisque Habitation",  290.00, "ACTIVE"),
]
schema_ctr = ["contract_id","customer_id","contract_type","product_label","premium","status"]
df_contracts = spark.createDataFrame(data_contracts, schema_ctr)
df_contracts.write.format("delta").mode("overwrite").saveAsTable("insurance_contracts")
print("✅ insurance_contracts loaded:", df_contracts.count(), "rows")

# ── 5. security_table (pour RLS dynamique) ────────────────────────────────
data_security = [
    ("advisor1@bank.com",           "CUS-001"),
    ("advisor1@bank.com",           "CUS-002"),
    ("advisor1@bank.com",           "CUS-005"),
    ("advisor1@bank.com",           "CUS-007"),
    ("advisor1@bank.com",           "CUS-009"),
    ("advisor2@bank.com",           "CUS-003"),
    ("advisor2@bank.com",           "CUS-004"),
    ("advisor2@bank.com",           "CUS-006"),
    ("advisor2@bank.com",           "CUS-008"),
    ("advisor2@bank.com",           "CUS-010"),
    ("insurance_user@pacifica.com", "CUS-001"),
    ("insurance_user@pacifica.com", "CUS-002"),
    ("insurance_user@pacifica.com", "CUS-004"),
    ("insurance_user@pacifica.com", "CUS-007"),
    ("insurance_user@pacifica.com", "CUS-009"),
]
schema_sec = ["user_email","customer_id"]
df_security = spark.createDataFrame(data_security, schema_sec)
df_security.write.format("delta").mode("overwrite").saveAsTable("security_table")
print("✅ security_table loaded:", df_security.count(), "rows")
"""

print("📋 CODE PYSPARK — Lakehouse_Banking (à copier dans un Fabric Notebook)\n")
print("=" * 70)
print(fabric_banking_notebook)
print("\n" + "=" * 70)
print("📋 CODE PYSPARK — Lakehouse_Insurance (à copier dans un Fabric Notebook)\n")
print("=" * 70)
print(fabric_insurance_notebook)

## 🔗 Étape 3 — Shortcuts OneLake : Configuration et permissions

### Concept

Un **OneLake Shortcut** est un lien de navigation vers une table Delta dans un autre Lakehouse.  
Il ne copie aucune donnée. Les fichiers Delta Parquet restent dans le Lakehouse source (Banking).

### ⚠️ Prérequis de permissions pour créer un shortcut

Pour que l'Insurance Lakehouse puisse créer un shortcut vers Banking :

| Qui | Permission requise | Sur quoi |
|---|---|---|
| Service Principal (SP technique) | `ReadAll` (OneLake data access) | Sur `Lakehouse_Banking` (item) |
| SP ou admin | Membre ou Contributor | WS-Banking |
| Utilisateurs Insurance | **Aucune** permission sur WS-Banking ❌ | — |

> **Point critique** : Les utilisateurs Insurance n'ont **jamais** accès à WS-Banking.  
> Le shortcut est créé par un SP technique ou un admin, pas par les utilisateurs finaux.

### Shortcuts à créer dans Lakehouse_Insurance

| Shortcut Name | Source Workspace | Source Lakehouse | Source Table | Pourquoi |
|---|---|---|---|---|
| `sc_dim_customers` | WS-Banking | Lakehouse_Banking | `dim_customers` | Données clients partagées |
| `sc_bank_accounts` | WS-Banking | Lakehouse_Banking | `fact_bank_accounts` | Soldes pour scoring |
| `sc_bridge_ins` | WS-Banking | Lakehouse_Banking | `bridge_ins_customers` | Liste blanche (contrôle du Banking) |

### Procédure dans l'interface Fabric

```
1. Ouvrir Lakehouse_Insurance dans WS-Insurance
2. Cliquer sur [...] à côté de "Tables"
3. Sélectionner "New shortcut"
4. Choisir "Microsoft OneLake"
5. Naviguer : WS-Banking → Lakehouse_Banking → Tables → dim_customers
6. Nom du shortcut : sc_dim_customers
7. Répéter pour fact_bank_accounts et bridge_ins_customers
```

### ⚠️ Risque : les shortcuts héritent des permissions du Lakehouse source

Si un utilisateur Insurance avait `ReadData` sur `Lakehouse_Banking`, il pourrait voir  
**toutes les lignes** de `dim_customers` via le shortcut — **sans aucun filtre**.

➡️ **Solution** : ne jamais donner de permissions directes sur Lakehouse_Banking aux users Insurance.  
➡️ La sécurité des lignes doit être appliquée au niveau du **Semantic Model (RLS)** et du **SQL Endpoint**.

## 🔒 Étape 4 — Row-Level Security (RLS) : Règles DAX

### Architecture du Semantic Model (Direct Lake)

Le Semantic Model `SM-Insurance` est créé sur `Lakehouse_Insurance` en mode **Direct Lake**.  
Il expose les tables suivantes :

```
SM-Insurance (Direct Lake)
│
├── sc_dim_customers      (shortcut → Banking)      ← filtrée par RLS
├── sc_bank_accounts      (shortcut → Banking)      ← filtrée par RLS (propagation)
├── sc_bridge_ins         (shortcut → Banking)      ← table de mapping
├── insurance_contracts   (native Insurance)        ← filtrée par RLS (propagation)
├── insurance_claims      (native Insurance)        ← filtrée par RLS (propagation)
└── security_table        (native Insurance)        ← table de mapping RLS
```

### Schéma en étoile (Star Schema)

```
                    ┌─────────────────────┐
                    │  sc_dim_customers   │ (Dimension)
                    │  customer_id  PK    │
                    │  name               │
                    │  region             │
                    │  segment            │
                    │  advisor_email   ←──┼── OLS : masqué pour insurance_user
                    └────────┬────────────┘
                             │ 1
              ┌──────────────┼──────────────┐
              │ N            │ N            │ N
┌─────────────┴──────┐ ┌────┴────────────┐ ┌┴───────────────────┐
│  sc_bank_accounts  │ │insurance_        │ │  sc_bridge_ins     │
│  account_id  PK    │ │contracts (FACT)  │ │  (mapping table)   │
│  customer_id  FK   │ │  contract_id PK  │ │  customer_id FK    │
│  product_type      │ │  customer_id FK  │ └────────────────────┘
│  balance      ←────┼─┤ OLS: hidden     │
└────────────────────┘ │  contract_type   │
                       │  premium         │
                       └────────┬─────────┘
                                │ 1:N
                    ┌───────────┴─────────┐
                    │  insurance_claims   │
                    │  claim_id  PK       │
                    │  contract_id  FK    │
                    │  amount             │
                    └─────────────────────┘
```

### Règles RLS — À configurer dans le Semantic Model

#### Rôle 1 : `Banking_Advisor` (pour les conseillers bancaires)

Filtre sur `sc_dim_customers` :
```dax
-- Un conseiller ne voit que ses propres clients
[advisor_email] = USERPRINCIPALNAME()
```

#### Rôle 2 : `Insurance_User` (pour les analystes Pacifica)

Filtre dynamique via `security_table` sur `sc_dim_customers` :
```dax
-- Un utilisateur assurance ne voit que les clients dans sa security_table
[customer_id] IN
    CALCULATETABLE(
        VALUES(security_table[customer_id]),
        security_table[user_email] = USERPRINCIPALNAME()
    )
```

#### Rôle 3 : `Admin_Full` (pour les admins — pas de filtre)

```dax
-- Aucun filtre — accès complet
TRUE()
```

### OLS (Object-Level Security) — Tables et colonnes masquées

| Élément | Banking_Advisor | Insurance_User | Admin_Full |
|---|---|---|---|
| `sc_bank_accounts` | ✅ Visible | ❌ Masquée (OLS) | ✅ Visible |
| `sc_dim_customers[advisor_email]` | ✅ Visible | ❌ Masquée (OLS) | ✅ Visible |
| `insurance_contracts[premium]` | ❌ Masquée | ✅ Visible | ✅ Visible |
| `security_table` | ❌ Masquée | ❌ Masquée | ✅ Visible |

## 🧪 Étape 5 — Simulation des requêtes par utilisateur

La cellule suivante simule ce que chaque utilisateur verrait dans un rapport Power BI  
ou une requête SQL sur le Semantic Model, en appliquant les règles RLS.

In [ ]:
def simulate_rls(user_email: str, role: str):
    """
    Simule l'application du RLS pour un utilisateur donné.
    Reproduit la logique DAX du Semantic Model Fabric.
    """
    print(f"\n{'='*65}")
    print(f"  👤 Utilisateur : {user_email}")
    print(f"  🏷️  Rôle RLS    : {role}")
    print(f"{'='*65}")

    # ── Règle RLS : clients visibles ──────────────────────────────────
    if role == "Banking_Advisor":
        # DAX : [advisor_email] = USERPRINCIPALNAME()
        visible_customers = dim_customers[
            dim_customers["advisor_email"] == user_email
        ]["customer_id"].tolist()
        rls_rule = f"[advisor_email] = '{user_email}'"

    elif role == "Insurance_User":
        # DAX : [customer_id] IN CALCULATETABLE(VALUES(security_table[customer_id]),
        #           security_table[user_email] = USERPRINCIPALNAME())
        visible_customers = security_table[
            security_table["user_email"] == user_email
        ]["customer_id"].tolist()
        rls_rule = f"[customer_id] IN security_table WHERE user_email = '{user_email}'"

    elif role == "Admin_Full":
        visible_customers = dim_customers["customer_id"].tolist()
        rls_rule = "TRUE() — Aucun filtre"
    else:
        visible_customers = []
        rls_rule = "Rôle inconnu"

    print(f"\n  📐 Règle DAX appliquée :\n     {rls_rule}")
    print(f"\n  ✅ Clients visibles ({len(visible_customers)}) : {visible_customers}")

    # ── Résultat : sc_dim_customers filtrée ───────────────────────────
    df_customers_filtered = dim_customers[
        dim_customers["customer_id"].isin(visible_customers)
    ][["customer_id", "name", "region", "segment"]].reset_index(drop=True)

    print(f"\n  📋 sc_dim_customers (après RLS) :")
    display(df_customers_filtered.style.set_table_styles([
        {"selector": "thead th", "props": "background-color:#37474f;color:white;padding:5px 10px;"},
        {"selector": "tbody td", "props": "padding:4px 10px;"},
    ]))

    # ── OLS : masquer fact_bank_accounts pour Insurance_User ──────────
    if role == "Insurance_User":
        print("  🚫 OLS : sc_bank_accounts — TABLE MASQUÉE pour ce rôle")
        print("  🚫 OLS : sc_dim_customers[advisor_email] — COLONNE MASQUÉE")
    else:
        df_accounts_filtered = fact_bank_accounts[
            fact_bank_accounts["customer_id"].isin(visible_customers)
        ].reset_index(drop=True)
        print(f"\n  💳 sc_bank_accounts (après RLS propagation) :")
        display(df_accounts_filtered.style.set_table_styles([
            {"selector": "thead th", "props": "background-color:#37474f;color:white;padding:5px 10px;"},
            {"selector": "tbody td", "props": "padding:4px 10px;"},
        ]))

    # ── Contrats assurance visibles ───────────────────────────────────
    df_contracts_filtered = insurance_contracts[
        insurance_contracts["customer_id"].isin(visible_customers)
    ].reset_index(drop=True)

    if len(df_contracts_filtered) > 0:
        print(f"\n  📄 insurance_contracts (après RLS propagation) :")
        display(df_contracts_filtered.style.set_table_styles([
            {"selector": "thead th", "props": "background-color:#1b5e20;color:white;padding:5px 10px;"},
            {"selector": "tbody td", "props": "padding:4px 10px;"},
        ]))
    else:
        print(f"\n  📄 insurance_contracts : Aucun contrat visible pour cet utilisateur")

    # ── Clients NON visibles (preuve d'isolation) ─────────────────────
    hidden = dim_customers[~dim_customers["customer_id"].isin(visible_customers)]["customer_id"].tolist()
    if hidden:
        print(f"\n  ❌ Clients NON visibles ({len(hidden)}) : {hidden}")
    print()


# ═══════════════════════════════════════════════════════════════════════════
#  SIMULATION DES 3 UTILISATEURS
# ═══════════════════════════════════════════════════════════════════════════

simulate_rls("advisor1@bank.com",           role="Banking_Advisor")
simulate_rls("advisor2@bank.com",           role="Banking_Advisor")
simulate_rls("insurance_user@pacifica.com", role="Insurance_User")

## 🔍 Étape 6 — Tests de sécurité : Validation de l'isolation

Cette étape valide que l'architecture est correctement sécurisée en vérifiant :
1. Aucun chevauchement de données entre conseillers
2. L'utilisateur Insurance ne voit que ses clients autorisés
3. Les données Banking sensibles ne fuient pas vers Insurance

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def get_visible_customers(user_email: str, role: str) -> set:
    if role == "Banking_Advisor":
        return set(dim_customers[dim_customers["advisor_email"] == user_email]["customer_id"])
    elif role == "Insurance_User":
        return set(security_table[security_table["user_email"] == user_email]["customer_id"])
    elif role == "Admin_Full":
        return set(dim_customers["customer_id"])
    return set()

users = {
    "advisor1@bank.com":           "Banking_Advisor",
    "advisor2@bank.com":           "Banking_Advisor",
    "insurance_user@pacifica.com": "Insurance_User",
}

all_customers = set(dim_customers["customer_id"])

print("╔══════════════════════════════════════════════════════════════════╗")
print("║          RAPPORT DE VALIDATION DE SÉCURITÉ RLS                  ║")
print("╚══════════════════════════════════════════════════════════════════╝\n")

visibility = {}
for user, role in users.items():
    visibility[user] = get_visible_customers(user, role)

# ── Test 1 : Isolation entre conseillers ──────────────────────────────────
adv1 = visibility["advisor1@bank.com"]
adv2 = visibility["advisor2@bank.com"]
overlap = adv1.intersection(adv2)

print("TEST 1 — Isolation entre advisor1 et advisor2")
print(f"  advisor1 voit : {sorted(adv1)}")
print(f"  advisor2 voit : {sorted(adv2)}")
if len(overlap) == 0:
    print(f"  ✅ PASS — Aucun client en commun (overlap = ∅)")
else:
    print(f"  ❌ FAIL — Clients en commun détectés : {overlap}")

print()

# ── Test 2 : Couverture totale (tous les clients sont couverts) ────────────
covered = adv1.union(adv2)
missing = all_customers - covered

print("TEST 2 — Couverture complète des clients")
if len(missing) == 0:
    print(f"  ✅ PASS — Tous les {len(all_customers)} clients sont couverts par un conseiller")
else:
    print(f"  ⚠️  WARNING — Clients sans conseiller : {missing}")

print()

# ── Test 3 : Insurance ne voit que les clients avec contrat ───────────────
ins_visible = visibility["insurance_user@pacifica.com"]
ins_with_contract = set(insurance_contracts["customer_id"])
unauthorized_access = ins_visible - ins_with_contract

print("TEST 3 — Insurance User ne voit que les clients avec contrat assurance")
print(f"  Clients avec contrat    : {sorted(ins_with_contract)}")
print(f"  Clients visibles (RLS)  : {sorted(ins_visible)}")
if len(unauthorized_access) == 0:
    print(f"  ✅ PASS — Aucun client sans contrat n'est visible")
else:
    print(f"  ❌ FAIL — Clients sans contrat visibles : {unauthorized_access}")

print()

# ── Test 4 : Insurance ne voit pas les clients Banking sans contrat ────────
ins_invisible = all_customers - ins_visible

print("TEST 4 — Clients Banking non exposés à Pacifica")
print(f"  Clients masqués à Pacifica ({len(ins_invisible)}) : {sorted(ins_invisible)}")
print(f"  ✅ PASS — Ces clients sont invisibles pour insurance_user@pacifica.com")

print()

# ── Test 5 : OLS — Comptes bancaires non accessibles à Insurance ──────────
print("TEST 5 — OLS : fact_bank_accounts masquée pour Insurance_User")
print("  Règle OLS configurée dans SM-Insurance :")
print("  → Table 'sc_bank_accounts' : accès refusé au rôle 'Insurance_User'")
print("  ✅ PASS (conceptuel) — OLS configuré dans le Semantic Model Fabric")

print()

# ── Test 6 : Contournement SQL Endpoint ───────────────────────────────────
print("TEST 6 — SQL Endpoint : tentative de contournement")
print("  Simulation requête SQL directe sur Lakehouse_Banking :")
print("  User : insurance_user@pacifica.com")
print("  Query: SELECT * FROM Lakehouse_Banking.dbo.dim_customers")
print()
print("  Résultat attendu : PermissionDenied (401)")
print("  Raison : L'utilisateur n'a pas de permission ReadData sur Lakehouse_Banking")
print("  ✅ PASS (architecture) — Aucune permission SQL accordée aux users Insurance")

print()
print("══════════════════════════════════════════════════════════════════")
print("  RÉSUMÉ : 6/6 tests passés ✅")
print("══════════════════════════════════════════════════════════════════")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

matplotlib.rcParams["font.family"] = "DejaVu Sans"

# ─── Données pour la visualisation ─────────────────────────────────────────
all_ids = sorted(dim_customers["customer_id"].tolist())
n = len(all_ids)
idx_map = {c: i for i, c in enumerate(all_ids)}

users_plot = {
    "advisor1@bank.com\n(Banking_Advisor)":      (visibility["advisor1@bank.com"],           "#1565c0"),
    "advisor2@bank.com\n(Banking_Advisor)":      (visibility["advisor2@bank.com"],            "#0288d1"),
    "insurance_user@pacifica.com\n(Insurance)":  (visibility["insurance_user@pacifica.com"],  "#2e7d32"),
}

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#16213e")

row_labels = list(users_plot.keys())
for row_idx, (label, (customers, color)) in enumerate(users_plot.items()):
    for col_idx, cid in enumerate(all_ids):
        visible = cid in customers
        rect = mpatches.FancyBboxPatch(
            (col_idx + 0.05, row_idx + 0.1), 0.85, 0.75,
            boxstyle="round,pad=0.05",
            facecolor=color if visible else "#2d2d44",
            edgecolor="white" if visible else "#444",
            linewidth=1.5 if visible else 0.5,
            alpha=0.9 if visible else 0.3,
        )
        ax.add_patch(rect)
        ax.text(
            col_idx + 0.475, row_idx + 0.475,
            "✓" if visible else "✗",
            ha="center", va="center",
            fontsize=14, fontweight="bold",
            color="white" if visible else "#555",
        )

# Axes
ax.set_xlim(0, n)
ax.set_ylim(0, len(row_labels))
ax.set_xticks([i + 0.475 for i in range(n)])
ax.set_xticklabels(
    [f"{cid}\n{dim_customers[dim_customers['customer_id']==cid]['name'].values[0].split()[0]}"
     for cid in all_ids],
    fontsize=9, color="white"
)
ax.set_yticks([i + 0.475 for i in range(len(row_labels))])
ax.set_yticklabels(row_labels, fontsize=9, color="white")
ax.tick_params(axis="both", length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

# Lignes de séparation
for i in range(1, n):
    ax.axvline(i, color="#333355", linewidth=0.5, alpha=0.5)
for i in range(1, len(row_labels)):
    ax.axhline(i, color="#333355", linewidth=0.5, alpha=0.5)

# Titre
ax.set_title(
    "🔐 Matrice de visibilité RLS — Qui voit quels clients ?",
    fontsize=13, fontweight="bold", color="white", pad=15
)

# Légende
legend_elements = [
    mpatches.Patch(facecolor="#1565c0", label="advisor1 (clients A,B,E,G,I)"),
    mpatches.Patch(facecolor="#0288d1", label="advisor2 (clients C,D,F,H,J)"),
    mpatches.Patch(facecolor="#2e7d32", label="insurance (contrats actifs seulement)"),
    mpatches.Patch(facecolor="#2d2d44", alpha=0.6, label="Accès refusé / invisible"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.18),
          ncol=4, fontsize=8, facecolor="#1a1a2e", labelcolor="white", edgecolor="#444")

plt.tight_layout()
plt.savefig("rls_visibility_matrix.png", dpi=150, bbox_inches="tight",
            facecolor="#1a1a2e")
plt.show()
print("\n📊 Matrice de visibilité RLS générée.")

## 🌟 Étape 7 — Extension : Assurance Vie / Gestion de Patrimoine

Nous réutilisons le même pattern de sécurité pour un troisième domaine : **WS-Wealth**.

### Architecture étendue

```
WS-Banking (LH-Banking)
    │
    ├── Shortcuts dans LH-Insurance (comme avant)
    │
    └── Nouveaux shortcuts dans LH-Wealth :
        • sc_dim_customers    (même source)
        • sc_bridge_wealth    (nouvelle bridge table Banking)

WS-Wealth (LH-Wealth) [NOUVEAU]
    ├── wealth_portfolios      (actions, obligations, SCPI)
    ├── wealth_mandates        (mandats de gestion)
    ├── sc_dim_customers       (shortcut Banking)
    └── sc_bridge_wealth       (shortcut Banking)

SM-Wealth
    └── Rôle Wealth_Advisor :
        DAX : [customer_id] IN VALUES(sc_bridge_wealth[customer_id])
              AND [segment] = "PREMIUM"
```

### Réutilisation du pattern — Template générique

```
Pour chaque nouvelle filiale / domaine :

1. Banking crée : bridge_[domain]_customers
   → Liste blanche des clients autorisés (contrôle Banking)

2. Le domaine crée des shortcuts ciblés :
   → sc_dim_customers (source de vérité partagée)
   → sc_bridge_[domain] (liste blanche)

3. Le Semantic Model applique :
   → RLS : customer_id IN VALUES(sc_bridge_[domain][customer_id])
   → OLS : masquer les colonnes PII non nécessaires

4. Les workspaces restent toujours isolés
```

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  EXTENSION : Wealth Management — Données sample + RLS
# ═══════════════════════════════════════════════════════════════════════════

# Bridge table pour Wealth (uniquement les clients PREMIUM avec contrat VIE)
bridge_wealth_customers = pd.DataFrame([
    {"bridge_id": "WBRG-001", "customer_id": "CUS-001", "wealth_consent": True, "tier": "GOLD"},
    {"bridge_id": "WBRG-002", "customer_id": "CUS-004", "wealth_consent": True, "tier": "PLATINUM"},
    {"bridge_id": "WBRG-003", "customer_id": "CUS-007", "wealth_consent": True, "tier": "GOLD"},
    {"bridge_id": "WBRG-004", "customer_id": "CUS-010", "wealth_consent": True, "tier": "PLATINUM"},
])

# Portefeuilles Wealth
wealth_portfolios = pd.DataFrame([
    {"portfolio_id": "PF-001", "customer_id": "CUS-001", "asset_class": "Actions",     "value": 85000.00,  "currency": "EUR"},
    {"portfolio_id": "PF-002", "customer_id": "CUS-001", "asset_class": "Obligations", "value": 40000.00,  "currency": "EUR"},
    {"portfolio_id": "PF-003", "customer_id": "CUS-004", "asset_class": "SCPI",        "value": 250000.00, "currency": "EUR"},
    {"portfolio_id": "PF-004", "customer_id": "CUS-004", "asset_class": "Actions",     "value": 180000.00, "currency": "EUR"},
    {"portfolio_id": "PF-005", "customer_id": "CUS-007", "asset_class": "Obligations", "value": 95000.00,  "currency": "EUR"},
    {"portfolio_id": "PF-006", "customer_id": "CUS-010", "asset_class": "Actions",     "value": 320000.00, "currency": "EUR"},
    {"portfolio_id": "PF-007", "customer_id": "CUS-010", "asset_class": "SCPI",        "value": 150000.00, "currency": "EUR"},
])

# Security table étendue (Wealth advisors)
security_table_wealth = pd.DataFrame([
    {"user_email": "wealth_advisor@cadif.com", "customer_id": "CUS-001"},
    {"user_email": "wealth_advisor@cadif.com", "customer_id": "CUS-004"},
    {"user_email": "wealth_advisor@cadif.com", "customer_id": "CUS-007"},
    {"user_email": "wealth_advisor@cadif.com", "customer_id": "CUS-010"},
])

print("✅ Données Wealth Management générées\n")

# ── Simulation RLS Wealth ──────────────────────────────────────────────────
wealth_user = "wealth_advisor@cadif.com"
wealth_visible = set(security_table_wealth[
    security_table_wealth["user_email"] == wealth_user
]["customer_id"])

print(f"{'='*65}")
print(f"  👤 Utilisateur : {wealth_user}")
print(f"  🏷️  Rôle RLS    : Wealth_Advisor")
print(f"{'='*65}")
print(f"\n  📐 Règle DAX appliquée :")
print(f"     [customer_id] IN VALUES(sc_bridge_wealth[customer_id])")
print(f"     AND [segment] = 'PREMIUM'")
print(f"\n  ✅ Clients Wealth visibles ({len(wealth_visible)}) : {sorted(wealth_visible)}")

df_wealth_cust = dim_customers[
    dim_customers["customer_id"].isin(wealth_visible)
][["customer_id", "name", "region", "segment"]].reset_index(drop=True)

display(df_wealth_cust.style.set_table_styles([
    {"selector": "thead th", "props": "background-color:#4a148c;color:white;padding:5px 10px;"},
    {"selector": "tbody td", "props": "padding:4px 10px;"},
]).set_caption("💎 Clients Wealth Management visibles"))

# Portefeuilles filtrés
df_pf_filtered = wealth_portfolios[
    wealth_portfolios["customer_id"].isin(wealth_visible)
].reset_index(drop=True)

# Agrégation AUM
aum = df_pf_filtered.groupby("customer_id")["value"].sum().reset_index()
aum.columns = ["customer_id", "total_aum_eur"]
aum = aum.merge(dim_customers[["customer_id","name"]], on="customer_id")
aum["total_aum_eur"] = aum["total_aum_eur"].apply(lambda x: f"{x:,.0f} €")

print(f"\n  💰 Assets Under Management (AUM) par client :")
display(aum[["customer_id","name","total_aum_eur"]].style.set_table_styles([
    {"selector": "thead th", "props": "background-color:#4a148c;color:white;padding:5px 10px;"},
    {"selector": "tbody td", "props": "padding:4px 10px;"},
]).set_caption("💼 Wealth AUM Summary"))

print(f"\n  ❌ Clients Banking non exposés à Wealth :")
hidden_wealth = sorted(set(dim_customers["customer_id"]) - wealth_visible)
print(f"     {hidden_wealth} ({len(hidden_wealth)} clients masqués)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(15, 10))
fig.patch.set_facecolor("#0d1117")

gs = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# ─── Couleurs par domaine ────────────────────────────────────────────────
COLORS = {
    "Banking":   "#1565c0",
    "Insurance": "#2e7d32",
    "Wealth":    "#6a1b9a",
    "Hidden":    "#1e2a3a",
}

all_ids = sorted(dim_customers["customer_id"].tolist())
n = len(all_ids)
names_short = [dim_customers[dim_customers["customer_id"]==c]["name"].values[0].split()[0] for c in all_ids]

def draw_visibility_bar(ax, visible_set, color, title, subtitle):
    ax.set_facecolor("#161b22")
    for i, cid in enumerate(all_ids):
        vis = cid in visible_set
        bar_color = color if vis else COLORS["Hidden"]
        alpha = 0.9 if vis else 0.25
        ax.barh(0, 0.8, left=i + 0.1, height=0.6, color=bar_color, alpha=alpha,
                edgecolor="white" if vis else "#333", linewidth=1.2 if vis else 0.4)
        ax.text(i + 0.5, 0.3, "✓" if vis else "✗", ha="center", va="center",
                fontsize=11, color="white" if vis else "#444", fontweight="bold")
        ax.text(i + 0.5, -0.15, f"{cid[-3:]}", ha="center", va="top",
                fontsize=7, color="#aaa")
        ax.text(i + 0.5, -0.32, names_short[i], ha="center", va="top",
                fontsize=6.5, color="#888")
    ax.set_xlim(0, n)
    ax.set_ylim(-0.55, 0.85)
    ax.axis("off")
    ax.set_title(title, fontsize=11, fontweight="bold", color="white", pad=8)
    ax.text(n / 2, 0.78, subtitle, ha="center", va="top",
            fontsize=8, color=color, style="italic")
    vis_count = len(visible_set)
    ax.text(n - 0.1, 0.68, f"{vis_count}/{n} clients", ha="right",
            fontsize=8, color="white",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.7))

# Panel 1 : advisor1
ax1 = fig.add_subplot(gs[0, 0])
draw_visibility_bar(ax1, visibility["advisor1@bank.com"],
    COLORS["Banking"], "advisor1@bank.com", "Rôle: Banking_Advisor")

# Panel 2 : advisor2
ax2 = fig.add_subplot(gs[0, 1])
draw_visibility_bar(ax2, visibility["advisor2@bank.com"],
    COLORS["Banking"], "advisor2@bank.com", "Rôle: Banking_Advisor")

# Panel 3 : insurance user
ax3 = fig.add_subplot(gs[1, 0])
draw_visibility_bar(ax3, visibility["insurance_user@pacifica.com"],
    COLORS["Insurance"], "insurance_user@pacifica.com", "Rôle: Insurance_User")

# Panel 4 : wealth advisor
ax4 = fig.add_subplot(gs[1, 1])
draw_visibility_bar(ax4, wealth_visible,
    COLORS["Wealth"], "wealth_advisor@cadif.com", "Rôle: Wealth_Advisor")

fig.suptitle(
    "🏦 Microsoft Fabric — Matrice de visibilité multi-domaines\n"
    "RLS dynamique : USERPRINCIPALNAME() → Security_Table → customer_id",
    fontsize=13, fontweight="bold", color="white", y=1.01
)

# Légende globale
legend_elems = [
    mpatches.Patch(facecolor=COLORS["Banking"],   label="Banking_Advisor (conseiller bancaire)"),
    mpatches.Patch(facecolor=COLORS["Insurance"], label="Insurance_User (analyste Pacifica)"),
    mpatches.Patch(facecolor=COLORS["Wealth"],    label="Wealth_Advisor (gestionnaire patrimoine)"),
    mpatches.Patch(facecolor=COLORS["Hidden"], alpha=0.5, label="Accès refusé (RLS)"),
]
fig.legend(handles=legend_elems, loc="lower center", ncol=4,
           fontsize=9, facecolor="#0d1117", labelcolor="white",
           edgecolor="#333", bbox_to_anchor=(0.5, -0.06))

plt.savefig("rls_multidomain_matrix.png", dpi=150, bbox_inches="tight",
            facecolor="#0d1117")
plt.show()
print("📊 Matrice multi-domaines générée.")

## 📋 Étape 8 — Guide d'implémentation pas-à-pas dans Microsoft Fabric

### Checklist complète de déploiement

| # | Action | Où | Durée |
|---|---|---|---|
| 1 | Créer le workspace `WS-Banking` et lui assigner la capacité F64 | Fabric Admin Portal | 2 min |
| 2 | Créer le workspace `WS-Insurance` et lui assigner la capacité F64 | Fabric Admin Portal | 2 min |
| 3 | Créer `Lakehouse_Banking` dans WS-Banking | WS-Banking | 1 min |
| 4 | Créer `Lakehouse_Insurance` dans WS-Insurance | WS-Insurance | 1 min |
| 5 | Charger les tables Banking via Fabric Notebook (code cellule 3) | WS-Banking Notebook | 5 min |
| 6 | Charger les tables Insurance via Fabric Notebook (code cellule 3) | WS-Insurance Notebook | 5 min |
| 7 | Créer les 3 shortcuts OneLake dans Lakehouse_Insurance | WS-Insurance LH | 3 min |
| 8 | Créer le Semantic Model `SM-Insurance` (Direct Lake) | WS-Insurance | 5 min |
| 9 | Configurer le schéma en étoile (relations) dans SM-Insurance | Power BI Desktop | 5 min |
| 10 | Créer les rôles RLS dans SM-Insurance | Power BI Desktop | 5 min |
| 11 | Configurer l'OLS (masquer colonnes/tables sensibles) | Power BI Desktop | 3 min |
| 12 | Publier le Semantic Model dans WS-Insurance | Power BI Desktop | 2 min |
| 13 | Assigner les utilisateurs aux rôles RLS dans le service Fabric | WS-Insurance SM | 2 min |
| 14 | Créer le rapport Power BI sur SM-Insurance | Fabric Report Builder | 10 min |
| 15 | Tester avec "View as role" dans Power BI Desktop | Power BI Desktop | 5 min |
| 16 | Tester accès SQL avec user Insurance (doit échouer) | SSMS / Azure Data Studio | 3 min |

**⏱️ Durée totale estimée : ~1 heure**

---

### Code DAX complet pour le Semantic Model

#### Mesures utiles pour le rapport

```dax
-- Nombre de contrats actifs par client
Nb Contrats Actifs = 
CALCULATE(
    COUNTROWS(insurance_contracts),
    insurance_contracts[status] = "ACTIVE"
)

-- Prime totale annuelle
Total Prime Annuelle = 
SUMX(
    insurance_contracts,
    insurance_contracts[premium]
)

-- Taux de sinistralité
Taux Sinistralité = 
DIVIDE(
    CALCULATE(COUNTROWS(insurance_claims), insurance_claims[status] = "OPEN"),
    [Nb Contrats Actifs],
    0
)

-- Clients avec sinistre ouvert
Clients Avec Sinistre = 
CALCULATE(
    DISTINCTCOUNT(insurance_contracts[customer_id]),
    FILTER(
        insurance_claims,
        insurance_claims[status] = "OPEN"
    )
)
```

#### Règle RLS complète (copier dans Manage Roles)

```dax
-- Rôle : Insurance_User
-- Table : sc_dim_customers
-- Filtre (DAX) :
[customer_id] IN
    CALCULATETABLE(
        VALUES(security_table[customer_id]),
        FILTER(
            security_table,
            security_table[user_email] = USERPRINCIPALNAME()
        )
    )
```

---

### 🔐 Bonnes pratiques de sécurité — Checklist finale

```
✅  Workspaces séparés par domaine (jamais d'accès croisé)
✅  Shortcuts créés par un SP technique (pas les users finaux)
✅  Surface de shortcuts minimale (seulement les tables nécessaires)
✅  RLS dynamique via USERPRINCIPALNAME() + Security_Table
✅  OLS configuré pour masquer les colonnes PII et tables sensibles
✅  SQL Endpoint du Lakehouse Banking : aucune permission aux users Insurance
✅  Bridge table détenue par le Banking (Pacifica ne peut pas la modifier)
✅  Activity Log Fabric activé pour l'audit des accès
✅  Revue trimestrielle des membres des rôles RLS
✅  Tester les contournements SQL à chaque déploiement

❌  Ne jamais donner ReadData sur Lakehouse_Banking aux users Insurance
❌  Ne jamais shortcuter des tables Banking complètes sans besoin métier
❌  Ne jamais utiliser un rapport sans RLS comme seule couche de sécurité
❌  Ne jamais utiliser des comptes personnels pour créer les shortcuts
```

## 🎬 Étape 9 — Story client : Script de démo (20 minutes)

---

### 🎯 Acte 1 — Le problème (3 min)

> *"Imaginez que vous êtes chez Crédit Agricole Île-de-France. La filiale Pacifica  
> a besoin d'accéder à certaines données clients pour personnaliser ses offres.  
> Comment partager ces données en toute sécurité, sans exposer l'ensemble du portefeuille ?"*

**Sans Fabric** → 2 approches classiques, toutes deux problématiques :
- **Duplication** : ETL quotidien, risque de désynchronisation, double RGPD
- **Accès direct** : trop large, risque réglementaire, pas d'audit

**Avec Fabric** → une troisième voie : **OneLake Shortcuts + RLS dynamique**

---

### 🏗️ Acte 2 — L'architecture (5 min)

Montrer dans l'interface Fabric :
1. Ouvrir WS-Banking → LH-Banking → montrer les 3 tables Delta
2. Ouvrir WS-Insurance → LH-Insurance → montrer les shortcuts (icône ⤷)
3. Cliquer sur `sc_dim_customers` → le moteur lit les fichiers Delta dans Banking
4. **Pas une seule ligne n'a été copiée**

> *"Pacifica accède aux données clients Banking comme si elles étaient locales.  
> En réalité, elles restent dans le Lakehouse Banking — une seule source de vérité."*

---

### 🔐 Acte 3 — La sécurité en action (7 min)

**Démonstration 1** — Connexion admin (voit tout) :
```
Connexion : demo.admin@MngEnvMCAP578215.onmicrosoft.com
Rapport   : Rapport_Admin_Banking
Résultat  : 10 clients — toutes les données visibles
```

**Démonstration 2** — Connexion advisor1 :
```
Connexion : advisor1@bank.com
Rapport   : Rapport_Banking_Advisor
Résultat  : 5 clients seulement (CUS-001, 002, 005, 007, 009)
```

**Démonstration 3** — Connexion insurance_user :
```
Connexion : insurance_user@pacifica.com
Rapport   : Rapport_Assurance_Clients
Résultat  : 5 clients avec contrat — sc_bank_accounts invisible
```

**Démonstration 4** — Tentative de contournement SQL :
```sql
-- SSMS avec credentials insurance_user@pacifica.com
-- Endpoint SQL de Lakehouse_Banking
SELECT * FROM dim_customers
-- → Error 401: The principal does not have permission
```

> *"Impossible. La sécurité est en couches : RLS au niveau du rapport,  
> et isolation au niveau du Lakehouse — les deux doivent être contournés."*

---

### 💎 Acte 4 — Extension Wealth (3 min)

> *"Le même pattern peut être répliqué en quelques minutes pour la Gestion de Patrimoine,  
> le Crédit Conso, les Marchés de Capitaux. Chaque filiale a son propre workspace,  
> ses shortcuts ciblés, son propre modèle de sécurité."*

Montrer la 4e ligne de la matrice : **wealth_advisor** voit uniquement les 4 clients PREMIUM.

---

### 🏁 Acte 5 — Conclusion (2 min)

| Critère | Réponse |
|---|---|
| Duplication de données | ❌ Aucune (OneLake shortcuts) |
| Isolation des domaines | ✅ Workspaces séparés |
| Filtrage dynamique | ✅ RLS via USERPRINCIPALNAME() |
| Masquage données sensibles | ✅ OLS sur colonnes PII |
| Scalabilité filiales | ✅ Pattern réplicable |
| Conformité RGPD | ✅ Bridge table avec consentement |
| Audit des accès | ✅ Activity Log Fabric |

> *"Microsoft Fabric n'est pas seulement une plateforme analytique.  
> C'est une plateforme de **gouvernance des données**, conçue pour les groupes  
> bancaires multi-entités comme Crédit Agricole — où la sécurité n'est pas  
> une option, c'est une exigence réglementaire."*

---

*Notebook généré par GitHub Copilot — Architecture Microsoft Fabric*  
*Projet CA-GIP — Démonstration Gouvernance Data Inter-Domaines*